# 03_procesa_gold

Construye las tablas Gold a partir de `silver_reportes`.

Responsabilidad:
- Actualizar `maestro_visitante`.
- Actualizar `visitas_mensuales`.
- Recalcular la fotografía mensual `visitante_mensual`.


In [0]:
# COMMAND ----------
dbutils.widgets.removeAll()

dbutils.widgets.text("catalog_name", "smart_data_dev", "Catalog name")
dbutils.widgets.text("schema_name", "storage", "Schema name")
dbutils.widgets.text("id_proceso", "02ae3d8f-05ac-4af0-86e8-05762dc2db18", "Process id")

catalog_name = dbutils.widgets.get("catalog_name").strip()
schema_name = dbutils.widgets.get("schema_name").strip()
id_proceso = dbutils.widgets.get("id_proceso").strip()

if not all([catalog_name, schema_name, id_proceso]):
    raise ValueError("catalog_name, schema_name e id_proceso son obligatorios.")

tabla_silver_sql = f"`{catalog_name}`.`{schema_name}`.`silver_reportes`"
tabla_maestro_sql = f"`{catalog_name}`.`{schema_name}`.`maestro_visitante`"
tabla_mensual_sql = f"`{catalog_name}`.`{schema_name}`.`visitas_mensuales`"
tabla_fotografia_sql = f"`{catalog_name}`.`{schema_name}`.`visitante_mensual`"

id_proceso_sql = id_proceso.replace("'", "''")

print(f"catalog_name: {catalog_name}")
print(f"schema_name: {schema_name}")
print(f"id_proceso: {id_proceso}")

In [0]:
# COMMAND ----------

fecha_ejecucion = spark.sql("SELECT current_timestamp() AS fecha_ejecucion").first()["fecha_ejecucion"]

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp_proceso AS
SELECT
    email,
    fecha_envio_mes AS periodo,
    MIN(CAST(fecha_envio AS DATE)) AS fecha_primera_visita_mes,
    MIN(CASE WHEN lower(trim(baja)) = 'si' THEN fecha_envio_mes END) AS periodo_baja
FROM {tabla_silver_sql}
WHERE id_proceso = '{id_proceso_sql}'
AND email IS NOT NULL
AND fecha_envio IS NOT NULL
GROUP BY email, fecha_envio_mes
""")

periodo_minimo = spark.sql("SELECT MIN(periodo) AS periodo_minimo FROM tmp_proceso").first()["periodo_minimo"]

if periodo_minimo is None:
    mensaje = f"No existen registros silver para procesar en gold. id_proceso={id_proceso}. Gold termina sin cambios."
    print(mensaje)
    dbutils.notebook.exit(mensaje)

print(f"Periodo mínimo a recalcular: {periodo_minimo}")

In [0]:
# COMMAND ----------

spark.sql(f"""
MERGE INTO {tabla_maestro_sql} AS tgt
USING (
    SELECT
        email,
        MIN(fecha_primera_visita_mes) AS fecha_primera_visita,
        MIN(periodo_baja) AS periodo_baja
    FROM tmp_proceso
    GROUP BY email
) AS src
ON tgt.email = src.email
WHEN MATCHED THEN UPDATE SET
    tgt.fecha_primera_visita = LEAST(tgt.fecha_primera_visita, src.fecha_primera_visita),
    tgt.periodo_baja = CASE
        WHEN tgt.periodo_baja IS NULL THEN src.periodo_baja
        WHEN src.periodo_baja IS NULL THEN tgt.periodo_baja
        ELSE LEAST(tgt.periodo_baja, src.periodo_baja)
    END,
    tgt.fecha_actualizacion = CAST('{fecha_ejecucion}' AS TIMESTAMP)
WHEN NOT MATCHED THEN INSERT (
    email,
    fecha_primera_visita,
    periodo_baja,
    fecha_actualizacion
)
VALUES (
    src.email,
    src.fecha_primera_visita,
    src.periodo_baja,
    CAST('{fecha_ejecucion}' AS TIMESTAMP)
)
""")

print("maestro_visitante actualizado.")

In [0]:
# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp_visitas_mensuales AS
SELECT
    s.fecha_envio_mes AS periodo,
    s.email,
    MIN(CAST(s.fecha_envio AS DATE)) AS fecha_primera_visita_mes,
    MAX(CAST(s.fecha_envio AS DATE)) AS fecha_ultima_visita_mes,
    COUNT(*) AS visitas_mes
FROM {tabla_silver_sql} AS s
WHERE EXISTS (
    SELECT 1
    FROM tmp_proceso AS p
    WHERE p.email = s.email
    AND p.periodo = s.fecha_envio_mes
)
GROUP BY s.email, s.fecha_envio_mes
""")

spark.sql(f"""
MERGE INTO {tabla_mensual_sql} AS tgt
USING tmp_visitas_mensuales AS src
ON tgt.email = src.email
AND tgt.periodo = src.periodo
WHEN MATCHED THEN UPDATE SET
    tgt.fecha_primera_visita_mes = src.fecha_primera_visita_mes,
    tgt.fecha_ultima_visita_mes = src.fecha_ultima_visita_mes,
    tgt.visitas_mes = src.visitas_mes,
    tgt.id_proceso = '{id_proceso_sql}',
    tgt.fecha_ejecucion = CAST('{fecha_ejecucion}' AS TIMESTAMP)
WHEN NOT MATCHED THEN INSERT (
    periodo,
    email,
    fecha_primera_visita_mes,
    fecha_ultima_visita_mes,
    visitas_mes,
    id_proceso,
    fecha_ejecucion
)
VALUES (
    src.periodo,
    src.email,
    src.fecha_primera_visita_mes,
    src.fecha_ultima_visita_mes,
    src.visitas_mes,
    '{id_proceso_sql}',
    CAST('{fecha_ejecucion}' AS TIMESTAMP)
)
""")

print("visitas_mensuales actualizado.")

In [0]:
# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp_corrida_recalculo AS
WITH limites AS (
    SELECT
        TO_DATE(CONCAT(MIN(periodo), '-01')) AS fecha_periodo_cero,
        TRUNC(CURRENT_DATE(), 'MONTH') AS fecha_periodo_actual
    FROM tmp_proceso
),
corrida_meses AS (
    SELECT DATE_FORMAT(fecha_mes, 'yyyy-MM') AS periodo
    FROM limites
    LATERAL VIEW EXPLODE(
        SEQUENCE(fecha_periodo_cero, fecha_periodo_actual, INTERVAL 1 MONTH)
    ) AS fecha_mes
),
maestro_filtrado AS (
    SELECT
        m.email,
        m.fecha_primera_visita,
        m.periodo_baja
    FROM {tabla_maestro_sql} AS m
)
SELECT
    univ.periodo,
    univ.email,
    univ.fecha_primera_visita,
    MAX(univ.fecha_ultima_visita) OVER (
        PARTITION BY univ.email
        ORDER BY univ.periodo
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS fecha_ultima_visita,
    SUM(univ.visitas_totales) OVER (
        PARTITION BY univ.email
        ORDER BY univ.periodo
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS visitas_totales,
    SUM(univ.visitas_anio_actual) OVER (
        PARTITION BY SUBSTRING(univ.periodo, 1, 4), univ.email
        ORDER BY univ.periodo
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS visitas_anio_actual,
    univ.visitas_mes_actual,
    CASE
        WHEN univ.periodo_baja IS NOT NULL AND univ.periodo >= univ.periodo_baja THEN univ.periodo_baja
        ELSE NULL
    END AS periodo_baja,
    univ.id_proceso,
    univ.fecha_ejecucion
FROM (
    SELECT
        co.periodo,
        ma.email,
        ma.fecha_primera_visita,
        mov.fecha_ultima_visita_mes AS fecha_ultima_visita,
        COALESCE(mov.visitas_mes, 0) AS visitas_totales,
        COALESCE(mov.visitas_mes, 0) AS visitas_anio_actual,
        COALESCE(mov.visitas_mes, 0) AS visitas_mes_actual,
        ma.periodo_baja AS periodo_baja,
        '{id_proceso_sql}' AS id_proceso,
        CAST('{fecha_ejecucion}' AS TIMESTAMP) AS fecha_ejecucion
    FROM corrida_meses AS co
    JOIN maestro_filtrado AS ma
        ON DATE_FORMAT(ma.fecha_primera_visita, 'yyyy-MM') <= co.periodo
    LEFT JOIN {tabla_mensual_sql} AS mov
        ON mov.email = ma.email
        AND mov.periodo = co.periodo

    UNION ALL

    SELECT
        vi.periodo,
        vi.email,
        vi.fecha_primera_visita,
        vi.fecha_ultima_visita,
        vi.visitas_totales,
        vi.visitas_anio_actual,
        CAST(0 AS BIGINT) AS visitas_mes_actual,
        vi.periodo_baja,
        vi.id_proceso,
        vi.fecha_ejecucion
    FROM {tabla_fotografia_sql} AS vi
    WHERE periodo = (
        SELECT DATE_FORMAT(ADD_MONTHS(TO_DATE(CONCAT(MIN(periodo), '-01')), -1), 'yyyy-MM')
        FROM tmp_proceso
    )
) AS univ
""")

limites = spark.sql("""
SELECT
    MIN(periodo) AS periodo_minimo,
    DATE_FORMAT(CURRENT_DATE(), 'yyyy-MM') AS periodo_actual
FROM tmp_proceso
""").first()

periodo_minimo = limites["periodo_minimo"]
periodo_actual = limites["periodo_actual"]

spark.sql(f"""
INSERT INTO {tabla_fotografia_sql}
BY NAME
REPLACE WHERE
    periodo >= '{periodo_minimo}'
    AND periodo <= '{periodo_actual}'
SELECT
    periodo,
    email,
    fecha_primera_visita,
    fecha_ultima_visita,
    visitas_totales,
    visitas_anio_actual,
    visitas_mes_actual,
    periodo_baja,
    id_proceso,
    fecha_ejecucion
FROM tmp_corrida_recalculo
WHERE periodo >= '{periodo_minimo}'
AND periodo <= '{periodo_actual}'
""")

print(f"Proceso gold terminado. id_proceso={id_proceso}; periodo_minimo={periodo_minimo}; periodo_actual={periodo_actual}")